In [0]:
import pandas as pd
import numpy as np

df = spark.table("workspace.gold.training_dataset_features_fifa").toPandas()

df["date"] = pd.to_datetime(df["date"])

print(df.shape)
display(df.head())

In [0]:
df_model = df[df["year"] >= 2000].copy()

print("Filas para modelado:", df_model.shape)
print(df_model["result"].value_counts())

In [0]:
features = [
    "tournament_weight",
    "is_world_cup",
    "is_world_cup_qualifier",
    "is_friendly",
    "is_neutral",

    "home_recent_games",
    "away_recent_games",

    "home_recent_win_rate",
    "away_recent_win_rate",
    "home_recent_draw_rate",
    "away_recent_draw_rate",
    "home_recent_loss_rate",
    "away_recent_loss_rate",

    "home_recent_avg_goals_for",
    "away_recent_avg_goals_for",
    "home_recent_avg_goals_against",
    "away_recent_avg_goals_against",
    "home_recent_avg_goal_diff",
    "away_recent_avg_goal_diff",

    "recent_win_rate_diff",
    "recent_draw_rate_diff",
    "recent_loss_rate_diff",
    "recent_avg_goals_for_diff",
    "recent_avg_goals_against_diff",
    "recent_avg_goal_diff_diff",
    "recent_games_diff",

    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "fifa_rank_diff",
    "fifa_points_diff"
]

target = "target"

df_model[features] = df_model[features].fillna(0)

X = df_model[features]
y = df_model[target]

print(X.shape)
print(y.value_counts())

In [0]:
train = df_model[df_model["date"] < "2022-01-01"].copy()
test = df_model[df_model["date"] >= "2022-01-01"].copy()

X_train = train[features].fillna(0)
y_train = train[target]

X_test = test[features].fillna(0)
y_test = test[target]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

mlp_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        batch_size=32,
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.2,
        n_iter_no_change=15,
        random_state=42
    ))
])

mlp_model.fit(X_train, y_train)

mlp_pred = mlp_model.predict(X_test)
mlp_proba = mlp_model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, mlp_pred))
print("Log Loss:", log_loss(y_test, mlp_proba, labels=[0, 1, 2]))

print(classification_report(
    y_test,
    mlp_pred,
    target_names=["home_win", "draw", "away_win"]
))

In [0]:
labels = ["home_win", "draw", "away_win"]

cm = confusion_matrix(y_test, mlp_pred, labels=[0, 1, 2])

cm_df = pd.DataFrame(
    cm,
    index=[f"Real {label}" for label in labels],
    columns=[f"Pred {label}" for label in labels]
)

display(cm_df)

In [0]:
base_results = spark.table("workspace.gold.model_results_base").toPandas()

mlp_results = pd.DataFrame([
    {
        "model": "Neural Network MLP",
        "accuracy": accuracy_score(y_test, mlp_pred),
        "log_loss": log_loss(y_test, mlp_proba, labels=[0, 1, 2])
    }
])

model_results_all = pd.concat([base_results, mlp_results], ignore_index=True)

display(model_results_all.sort_values("log_loss"))

In [0]:
model_results_all_spark = spark.createDataFrame(model_results_all)

model_results_all_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold.model_results_all")

print("Resultados de modelos guardados en Gold.")

In [0]:
test_predictions = test[[
    "date",
    "home_team",
    "away_team",
    "tournament",
    "result",
    "target"
]].copy()

test_predictions["predicted_class"] = mlp_pred
test_predictions["prob_home_win"] = mlp_proba[:, 0]
test_predictions["prob_draw"] = mlp_proba[:, 1]
test_predictions["prob_away_win"] = mlp_proba[:, 2]

class_mapping_reverse = {
    0: "home_win",
    1: "draw",
    2: "away_win"
}

test_predictions["predicted_result"] = test_predictions["predicted_class"].map(class_mapping_reverse)

display(test_predictions.head())

In [0]:
test_predictions_spark = spark.createDataFrame(test_predictions)

test_predictions_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold.neural_network_test_predictions")

print("Predicciones de red neuronal guardadas en Gold.")